# Example 7.11 / 2: Covering, Packing and Partitioning Rows

Companion notebook for **7.11 Knapsack, Covering and Packing** in the AMPL lab Documentation, section "Covering, Packing and Partitioning Rows". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Covering, Packing and Partitioning Rows

For an incidence parameter `covers[i,j]` equal to 1 when column `i` serves requirement `j`:

| Family | Core row for each requirement `j` | Meaning |
|---|---|---|
| set covering | `sum {i in COLS} covers[i,j] * Take[i] >= 1` | every requirement served at least once |
| set packing | `sum {i in COLS} covers[i,j] * Take[i] <= 1` | no requirement served twice |
| set partitioning | `sum {i in COLS} covers[i,j] * Take[i] = 1` | every requirement served exactly once |

The index roles matter: `i` denotes a selectable column and `j` a requirement row. The incidence
coefficient should usually be binary; if it is a capacity or multiplicity instead, the row has a
different interpretation. Costs normally belong on selected columns.

Here four candidate stations cover five districts; open the cheapest set of stations that covers
every district:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set COLS;                         # candidate stations
set ROWS;                         # districts to cover
param cost {COLS} >= 0;           # cost of opening the station
param covers {COLS, ROWS} binary; # 1 if the station covers the district
var Take {COLS} binary;           # 1 if the station is opened

minimize Cost: sum {i in COLS} cost[i] * Take[i];
# covering row: every district has at least one open station
subject to Cover {j in ROWS}: sum {i in COLS} covers[i,j] * Take[i] >= 1;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["COLS"] = ["s1", "s2", "s3", "s4"]
ampl.set["ROWS"] = ["d1", "d2", "d3", "d4", "d5"]
ampl.param["cost"] = {"s1": 5, "s2": 4, "s3": 3, "s4": 6}
served = {"s1": ["d1", "d2"], "s2": ["d2", "d3", "d4"], "s3": ["d4", "d5"], "s4": ["d1", "d3", "d5"]}
ampl.param["covers"] = {(i, j): int(j in served[i]) for i in served
                        for j in ["d1", "d2", "d3", "d4", "d5"]}

In [5]:
# run.py
ampl.solve(solver="highs")
print([i for i, v in ampl.var["Take"].to_dict().items() if v > 0.5])

HiGHS 1.14.0: optimal solution; objective 10
0 simplex iterations
1 branching nodes
['s2', 's4']


### Pure AMPL: data.dat and model.run

Stations `s2` and `s4` together cover all five districts for 10; no cheaper pair or triple does.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set COLS := s1 s2 s3 s4;
set ROWS := d1 d2 d3 d4 d5;
param cost := s1 5  s2 4  s3 3  s4 6;
param covers:  d1 d2 d3 d4 d5 :=
         s1     1  1  0  0  0
         s2     0  1  1  1  0
         s3     0  0  0  1  1
         s4     1  0  1  0  1 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Take;

HiGHS 1.14.0: optimal solution; objective 10
0 simplex iterations
1 branching nodes
Take [*] :=
s1  0
s2  1
s3  0
s4  1
;

